In [ ]:
from torchvision import datasets
from torchvision import transforms as T
from torch.utils.data import DataLoader, Subset
import torch
import torch.nn as nn
from torchinfo import summary
import matplotlib.pyplot as plt
import os
import torch.optim as optim
from tqdm import tqdm
from sklearn.model_selection import train_test_split


In [ ]:
import sys
sys.path.append("..")

from model import ModifiedMobileNetV1

In [ ]:
USE_GRAYSCALE = False
BATCH_SIZE = 8
IMAGE_SIZE = 96 
IN_CHANNELS = 1 if USE_GRAYSCALE else 3 
NUM_CLASSES = 1
ALPHA = 0.25
NUM_EPOCHS = 100
LEARNING_RATE = 2e-3
L2_VALUE = 4e-6

In [ ]:
if USE_GRAYSCALE:
    transform = T.Compose([
        # ImageFolder will treat images as RGB images, even if they're stored as grayscale.
        T.Grayscale(),
        T.ToTensor(),
    ])
else:
    transform = T.Compose([
        T.ToTensor(),
    ])

data_dir = "../data/subset"
dataset = datasets.ImageFolder(data_dir, transform=transform)

In [ ]:

train_idx, val_idx = train_test_split(
    range(len(dataset)),
    test_size=0.2,
    stratify=dataset.targets,
    random_state=123
)

train_subset = Subset(dataset, train_idx)
val_subset = Subset(dataset, val_idx)

train_loader = DataLoader(train_subset, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_subset, batch_size=8, shuffle=True, num_workers=2)

In [ ]:
print(dataset.classes, dataset.class_to_idx)

In [ ]:
for imgs, labels in train_loader:
    print(f"input shape: {imgs.shape}, label shape: {labels.shape}")
    print(f"input dtype: {imgs[0].dtype}, label dtype: {labels[0].dtype}")
    print(f"label sample: {labels}")
    break

Architecture unit tests (individual layer shape checks) that lived here
during initial development have moved to `model.py`'s own self-test
(`python model.py`), now that the architecture is extracted into its own file.

In [ ]:
model = ModifiedMobileNetV1(alpha=ALPHA, num_classes=NUM_CLASSES, in_channels=IN_CHANNELS)
x = torch.randn((BATCH_SIZE, IN_CHANNELS, IMAGE_SIZE, IMAGE_SIZE))
model(x)

In [ ]:
model = ModifiedMobileNetV1(alpha=ALPHA, num_classes=NUM_CLASSES, in_channels=IN_CHANNELS)
summary(model, input_size=(1, IN_CHANNELS, IMAGE_SIZE, IMAGE_SIZE))

In [ ]:
def training(model, train_loader, val_loader, num_epochs, learning_rate, l2_value, threshold):
    
    loss_fn = nn.BCEWithLogitsLoss()

    # Values are adapted from Wake Vision paper.
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=l2_value)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    train_losses =  []
    val_losses = []
    val_accuracies, val_precisions, val_recalls, val_f1s = [], [], [], []
    least_val_loss = float("inf")
    os.makedirs("../models", exist_ok=True)

    for epoch in range(num_epochs):

        model.train()
        train_loss = 0.0
        tp = fp = tn = fn = 0


        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False)
        for imgs, y_actual in progress_bar:
            y_pred = model(imgs)
            loss = loss_fn(y_pred.squeeze(1), y_actual.float())
            probs = torch.sigmoid(y_pred.squeeze(1))
            preds = (probs >= threshold).long()
            y_actual = y_actual.long()

            tp += ((preds == 1) & (y_actual == 1)).sum().item()
            fp += ((preds == 1) & (y_actual == 0)).sum().item()
            tn += ((preds == 0) & (y_actual == 0)).sum().item()
            fn += ((preds == 0) & (y_actual == 1)).sum().item()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        scheduler.step()
        train_acc = (tp+tn) / (tp+fp+tn+fn)
        avg_train_loss = train_loss / len(train_loader)
        train_losses.append(avg_train_loss)
        tqdm.write(f"Epoch {epoch+1}\nTrain -> Loss: {avg_train_loss:.4f}, TrainAcc: {train_acc:.4f}")

        model.eval()
        val_loss = 0.0
        tp = fp = tn = fn = 0
        with torch.no_grad():
            for imgs, y_actual in val_loader:
                y_pred = model(imgs)
                loss = loss_fn(y_pred.squeeze(1), y_actual.float())
                probs = torch.sigmoid(y_pred.squeeze(1))
                preds = (probs >= threshold).long()
                y_actual = y_actual.long()

                tp += ((preds == 1) & (y_actual == 1)).sum().item()
                fp += ((preds == 1) & (y_actual == 0)).sum().item()
                tn += ((preds == 0) & (y_actual == 0)).sum().item()
                fn += ((preds == 0) & (y_actual == 1)).sum().item()
                val_loss += loss.item()

            
            accuracy = (tp+tn) / (tp+fp+tn+fn)
            precision = tp / (tp+fp) if (tp+fp) else 0.0
            recall = tp / (tp+fn) if (tp+fn) else 0.0
            f1 = 2*precision*recall / (precision+recall) if (precision+recall) else 0.0

            avg_loss = val_loss / len(val_loader)
            val_losses.append(avg_loss)
            val_accuracies.append(accuracy)
            val_precisions.append(precision)
            val_recalls.append(recall)
            val_f1s.append(f1)
            tqdm.write(f"Validation -> Loss: {avg_loss:.4f}, Acc: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}\n ------------------")

            if(avg_loss < least_val_loss):
                least_val_loss = avg_loss
                torch.save(model.state_dict(), "../models/best_baseline_model.pth")
                tqdm.write("Best model saved.") 

    return model, train_losses, val_losses, val_accuracies, val_precisions, val_recalls, val_f1s 

In [ ]:
model = ModifiedMobileNetV1(alpha=ALPHA, num_classes=NUM_CLASSES, in_channels=IN_CHANNELS)
trained_model, train_losses, val_losses, val_accuracies, val_precisions, val_recalls, val_f1s = training(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    l2_value=L2_VALUE,
    threshold=0.5)

**First full train/val run — overfitting confirmed early.** Best validation
loss (0.6666, F1 0.602) occurred at epoch 5 of 100; by epoch 100, train
accuracy reached 0.996 while validation loss rose to 1.978 and F1 fell to
0.585. The model stopped generalizing almost immediately and spent the
remaining epochs memorizing the training set. Confirms 480 training images
is far too little for this model capacity.

*(Note: no random seed is set, so re-running this cell will produce different
exact numbers each time — the overfitting pattern is consistent, the specific
epoch/loss values are not.)*

In [ ]:
epochs = range(1, NUM_EPOCHS + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(epochs, train_losses, label="Train Loss")
ax1.plot(epochs, val_losses, label="Val Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("BinaryCrossEntropy Loss")
ax1.set_title("Loss")
ax1.legend()

ax2.plot(epochs, val_accuracies, label="Accuracy")
ax2.plot(epochs, val_precisions, label="Precision")
ax2.plot(epochs, val_recalls, label="Recall")
ax2.plot(epochs, val_f1s, label="F1")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Score")
ax2.set_ylim(0, 1)
ax2.set_title("Validation metrics")
ax2.legend()

plt.tight_layout()
plt.show()